### No-show Prediction - Random Forest

In [0]:
from features import engineering

In [0]:
import os, yaml

In [0]:

config_file = "../config.yaml" if os.path.exists("../config.yaml") else "config.example.yaml"

with open(config_file) as f:
    cfg = yaml.safe_load(f)
train_scaled_path = cfg["train_scaled_dataset_path"]
val_scaled_path = cfg["val_scaled_dataset_path"]
test_scaled_path = cfg["test_scaled_dataset_path"]
volume_path = cfg["volume_path"]
logistic_regression_model_path = cfg["lr_model_path"]
rf_model_path = cfg["rf_model_path"]

# Load leak-free 60/20/20 splits (encoders & scalers fit on train only)
train_scaled = spark.table(train_scaled_path)
val_scaled = spark.table(val_scaled_path)
test_scaled = spark.table(test_scaled_path)

In [0]:
print(f"Features read in from Delta Table")
print(f"  Train: {train_scaled.count():,} rows, {len(train_scaled.columns)} columns")
print(f"  Val:   {val_scaled.count():,} rows, {len(val_scaled.columns)} columns")
print(f"  Test:  {test_scaled.count():,} rows, {len(test_scaled.columns)} columns")

### RFE

In [0]:
# RFE requested 12 features but scaledFeatures now has only 10
# (cyclical dates + Neighborhood_te + Gender_vec).
# Skip RFE and use all 10 features.

# Feature names in the order they were assembled into "scaledFeatures"
selected_features = [
    "Sched_month_sin", "Sched_month_cos",
    "Sched_dayofyear_sin", "Sched_dayofyear_cos",
    "Appoi_month_sin", "Appoi_month_cos",
    "Appoi_dayofyear_sin", "Appoi_dayofyear_cos",
    "Neighborhood_te", "Gender_vec",
]

print(f"Using all {len(selected_features)} features (RFE skipped):")
for i, f in enumerate(selected_features):
    print(f"  {i}: {f}")

In [0]:
from pyspark.ml.feature import VectorSlicer

# Using all 10 features (RFE skipped), so slice indices 0-9
selected_indices = list(range(len(selected_features)))
print(f"Selected feature indices: {selected_indices}")
print(f"Selected features: {selected_features}")

# Slice scaledFeatures to create selectedFeatures (all 10 features)
slicer = VectorSlicer(
    inputCol="scaledFeatures",
    outputCol="selectedFeatures",
    indices=selected_indices
)

train_scaled = slicer.transform(train_scaled)
val_scaled = slicer.transform(val_scaled)
test_scaled = slicer.transform(test_scaled)

#### Hyperparameter tuning

In [0]:
import numpy as np
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator, CrossValidatorModel

In [0]:
# Scorers
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# AUC evaluator — threshold-independent, used for reporting only
auc_evaluator = BinaryClassificationEvaluator(
    labelCol = "Showed_up",
    metricName = "areaUnderROC"
)

# F1 evaluator — threshold-dependent, used for model selection during CV
# Aligns CV optimization with the business metric (balanced precision/recall)
cv_evaluator = MulticlassClassificationEvaluator(
    labelCol = "Showed_up",
    predictionCol = "prediction",
    metricName = "f1"
)


### CrossValidator Documentation
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html

In [0]:
os.environ["SPARKML_TEMP_DFS_PATH"] = volume_path #"/Volumes/workspace/default/no_show_volume"

### Random Forest Classifier

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol = 'selectedFeatures',
                            labelCol = 'Showed_up',
                            predictionCol = 'prediction',
                            weightCol = 'weightCol'
)

In [0]:
%skip
# 3 x 2 x 2 = 12 combos x 3 folds = 36 models
rf.setNumTrees(100)

rf_param_grid = (ParamGridBuilder()
    .addGrid(rf.maxDepth, [10, 12, 15])
    .addGrid(rf.minInstancesPerNode, [5, 20])
    .addGrid(rf.featureSubsetStrategy, ["sqrt", "log2"])
    .build())

In [0]:
rf.setNumTrees(10)

rf_param_grid = (ParamGridBuilder()
    .addGrid(rf.maxDepth, [10])
    .addGrid(rf.minInstancesPerNode, [20])
    .addGrid(rf.featureSubsetStrategy, ["sqrt"])
    .build())

### HTuning notes

Changing maxDepth from (5,10) to (5,15) increased f1 by 13%. Chnging numtrees to 75,150 was negligible.

In [0]:
# Clear cached ML models to avoid ML_CACHE_SIZE_OVERFLOW_EXCEPTION on Serverless
# spark.catalog.clearCache() clears SQL cache, NOT the ML model cache.
# On Spark Connect, fitted models are cached in the session (1GB limit).
# Delete ALL model-related objects and force GC before fitting a new one.
import gc

for name in ['rf_model', 'rf_predictions', 'cv', 'cv_model', 'best_model', 'pipeline_model']:
    globals().pop(name, None)

# clear IPython output history that may hold model refs
get_ipython().run_line_magic('reset', '-f out')
gc.collect()

#### RF CrossValidator

In [0]:
from pyspark.ml.tuning import CrossValidator
cv = CrossValidator(estimator=rf, estimatorParamMaps=rf_param_grid,
                     evaluator=cv_evaluator, numFolds=3, parallelism=4)

In [0]:
train_df = spark.read.table('train_df_tbl')
test_df = spark.read.table('test_df_tbl')

In [0]:
# Don't collect sub-models — 36 combos × 3 folds = 108 models would overflow
# the 1GB Spark Connect ML session cache. Only keep the best model.
cv.setCollectSubModels(False)

# train_scaled already has indexing/encoding applied, fit RF directly
rf_model = cv.fit(train_scaled)
rf_val_predictions = rf_model.transform(val_scaled)
rf_predictions = rf_model.transform(test_scaled)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F

In [0]:
import mlflow       # Experiment tracking & Record ML runs
import mlflow.spark # Log model artifacts
import os
from mlflow.models import infer_signature

os.environ["MLFLOW_DFS_TMP"] = volume_path

In [0]:
# Multiclass evaluator for weighted precision, recall, F1, accuracy
multi_evaluator = MulticlassClassificationEvaluator(
    predictionCol = 'prediction',
    labelCol = 'Showed_up',
    metricName = 'f1'
)

### Precision-Recall AUC

PR-AUC (Average Precision) summarizes the precision-recall curve into a single threshold-independent metric. Unlike ROC-AUC, it is sensitive to class imbalance and is more informative when the positive class is rare or when the business cost of false positives vs. false negatives is asymmetric.

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import matplotlib.pyplot as plt
import numpy as np

# --- PR-AUC (threshold-independent, Spark-native) ---
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="Showed_up",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)
pr_auc = pr_evaluator.evaluate(rf_predictions)
print(f"PR-AUC (Spark evaluator): {pr_auc:.4f}")

# --- PR Curve visualization ---
# Extract positive-class probability; also used by downstream threshold-tuning cell
get_prob_1 = F.udf(lambda v: float(v[1]), DoubleType())
rf_predictions = rf_predictions.withColumn("prob_1", get_prob_1("probability"))
rf_val_predictions = rf_val_predictions.withColumn("prob_1", get_prob_1("probability"))

# Collect label + score for plotting (test set only)
pdf = rf_predictions.select("Showed_up", "prob_1").toPandas()
y_true = pdf["Showed_up"].values.astype(int)
y_scores = pdf["prob_1"].values

# Sweep thresholds to build PR curve
thresholds = np.linspace(0.0, 1.0, 101)
precisions, recalls = [], []
for t in thresholds:
    y_pred = (y_scores >= t).astype(int)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    p = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precisions.append(p)
    recalls.append(r)

# Trapezoidal sanity check against Spark's PR-AUC (sort by recall for correct sign)
order = np.argsort(recalls)
trapz_pr_auc = float(np.trapezoid(np.array(precisions)[order], np.array(recalls)[order]))
print(f"PR-AUC (trapezoidal):     {trapz_pr_auc:.4f}")

# Plot PR curve
fig, ax = plt.subplots(figsize=(7, 5))
ax.step(recalls, precisions, where='post', color='steelblue', linewidth=2)
ax.fill_between(recalls, precisions, step='post', alpha=0.2, color='steelblue')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'Precision-Recall Curve (PR-AUC = {pr_auc:.4f})')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
display(fig)

In [0]:
signature = infer_signature(train_df, rf_predictions)

### Evaluate and log RF metrics

In [0]:
from pyspark.sql import functions as F

# --- Threshold tuning on VALIDATION set ---
# Sweep thresholds on validation predictions to find the best F1.
# The test set is held out and only evaluated at the end with the tuned threshold.
best_threshold = 0.5
best_f1 = 0.0
print("Threshold sweep on VALIDATION set (weighted F1):")
for threshold in [i / 100.0 for i in range(30, 71, 5)]:
    temp_pred = rf_val_predictions.withColumn(
        "temp_prediction",
        F.when(F.col("prob_1") >= threshold, 1.0).otherwise(0.0)
    )
    temp_f1 = multi_evaluator.evaluate(
        temp_pred,
        {multi_evaluator.predictionCol: "temp_prediction",
         multi_evaluator.metricName: "f1"}
    )
    print(f"  threshold={threshold:.2f}  F1={temp_f1:.4f}")
    if temp_f1 > best_f1:
        best_f1 = temp_f1
        best_threshold = threshold

print(f"\nBest threshold (from validation): {best_threshold:.2f} (F1={best_f1:.4f})")

# Apply best threshold to the TEST set
rf_predictions = rf_predictions.withColumn(
    "prediction",
    F.when(F.col("prob_1") >= best_threshold, 1.0).otherwise(0.0)
)

# --- Weighted metrics on TEST set (tuned threshold) ---
precision = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "weightedPrecision"})
recall = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "weightedRecall"})
f1 = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "f1"})
accuracy = multi_evaluator.evaluate(rf_predictions, {multi_evaluator.metricName: "accuracy"})

# --- AUC (threshold-independent; PR-AUC computed in PR-AUC cell above) ---
auc = auc_evaluator.evaluate(rf_predictions)

# --- Per-class metrics ---
confusion = rf_predictions.groupBy("Showed_up", "prediction").count().collect()
cm = {}
for row in confusion:
    cm[(float(row['Showed_up']), float(row['prediction']))] = row['count']

def class_metrics(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_val = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1_val

# Class 1 (Showed up)
p_1, r_1, f1_1 = class_metrics(
    cm.get((1.0, 1.0), 0), cm.get((0.0, 1.0), 0), cm.get((1.0, 0.0), 0)
)
# Class 0 (No-show)
p_0, r_0, f1_0 = class_metrics(
    cm.get((0.0, 0.0), 0), cm.get((1.0, 0.0), 0), cm.get((0.0, 1.0), 0)
)

In [0]:
# Use the signature already inferred in cell 70 (train_df → rf_predictions)
# input_example would fail here due to non-JSON-serializable Vector columns

mlflow.set_experiment(cfg["mlflow"]["experiment_path"])

with mlflow.start_run(run_name="RandomForestModel") as run:
    # Log model artifact with pre-computed signature
    mlflow.spark.log_model(rf_model, 
                           "random_forest_model",
                           signature=signature  # From cell 70
    ) # End log model
    mlflow.log_param("pip_requirements", ["pyspark==4.1.0"])
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("featuresCol", "features")
    mlflow.log_param("labelCol", "Showed_up")
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("best_threshold", best_threshold)
    mlflow.log_metric("f1_class1_showed_up", f1_1)
    mlflow.log_metric("f1_class0_no_show", f1_0)
    mlflow.log_param("run_description", "")

### Log Hyperparameters

In [0]:
# Get the model URI from the current run
model_uri = f"runs:/{run.info.run_id}/random_forest_model"

# Register to Model Registry (creates a new version automatically)
model_details = mlflow.register_model(
    model_uri=model_uri,
    name="noshows_random_forest"
)

print(f"Registered model version: {model_details.version}")

In [0]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
   name = rf_model_path,
    alias="champion",
   version=model_details.version
)

In [0]:
# AUC and accuracy are now computed in the evaluation cell with tuned threshold

print("=" * 50)
print("  Random Forest — Run Summary")
print("=" * 50)
print()
print("RFE Selected Features:")
for f in selected_features:
    print(f"  • {f}")
print()
print(f"Best numTrees:        {rf_model.bestModel.getNumTrees}")
print(f"Best maxDepth:         {rf_model.bestModel.getMaxDepth()}")
print(f"Best threshold:        {best_threshold:.2f}")
print()
print("Evaluation Metrics (test set, tuned threshold):")
print(f"  ROC-AUC:   {auc:.4f}")
print(f"  PR-AUC:    {pr_auc:.4f}")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  F1:        {f1:.4f}  (weighted)")
print(f"  Precision: {precision:.4f}  (weighted)")
print(f"  Recall:    {recall:.4f}  (weighted)")
print(f"  No-show class recall (Class 0): {r_0:.4f}")
print()
print("Per-class Metrics:")
print(f"  Class 1 (Showed up):  P={p_1:.4f}  R={r_1:.4f}  F1={f1_1:.4f}")
print(f"  Class 0 (No-show):    P={p_0:.4f}  R={r_0:.4f}  F1={f1_0:.4f}")
print()
print(f"Registered model:    {rf_model_path}")
print(f"Model version:       {model_details.version}")
print(f"Alias:               champion")
print("=" * 50)